In [ ]:
import numpy as np
import pandas as pd
from glob import glob
from matplotlib import pyplot as plt
import seaborn as sns
import pickle as pkl
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from one.api import ONE
from sklearn.decomposition import PCA
from dPCA import dPCA
from sklearn.metrics import euclidean_distances
from scipy import stats
from manifold.pseudosession_manifolds import process_eids

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import plotly.io as pio

pio.renderers.default = "notebook"

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
def get_region_stats():
    one = ONE()
    units_df = bwm_units(one)
    neuron_counts = units_df.groupby(["Beryl", "eid"]).size().reset_index(name="neuron_count")
    valid_pairings = neuron_counts[neuron_counts["neuron_count"] >= 5]
    # print(valid_pairings)  # atleast 10 neurons
    final_counts = (
        valid_pairings.groupby("Beryl")["eid"]
        .nunique()
        .reset_index(name="valid_eid_count")
        .sort_values(by="valid_eid_count", ascending=False)
    )
    # print(final_counts)
    regions_of_interest = final_counts[final_counts["valid_eid_count"] >= 20]["Beryl"].values
    region_totals = units_df.groupby("Beryl").size()
    valid_regions = region_totals[region_totals >= 20].index
    df_valid = units_df[units_df["Beryl"].isin(valid_regions)]

    final_table = (
        df_valid.groupby("Beryl")["eid"]
        .agg(
            total_neurons="size",
            unique_eid_count="nunique",
            eids_list=lambda x: list(x.unique()),
        )
        .reset_index()
        .sort_values(by="total_neurons", ascending=False)
    )
    # final_table[final_table["unique_eid_count"] >= 10]["Beryl"].values
    return final_table, final_counts

In [6]:
def plot_trajectories(traj_A, traj_B):
    import plotly.graph_objects as go

    trace_A = go.Scatter3d(
        x=traj_A[:, 0],
        y=traj_A[:, 1],
        z=traj_A[:, 2],
        mode="lines+markers",
        line=dict(color="blue", width=4),
        marker=dict(
            size=[8] + [2] * (len(traj_A) - 1), color="blue"
        ),  # Makes the first dot bigger
        name="Correct",
    )

    trace_B = go.Scatter3d(
        x=traj_B[:, 0],
        y=traj_B[:, 1],
        z=traj_B[:, 2],
        mode="lines+markers",
        line=dict(color="red", width=4),
        marker=dict(size=[8] + [2] * (len(traj_B) - 1), color="red"),  # Makes the first dot bigger
        name="Incorrect",
    )

    fig = go.Figure(data=[trace_A, trace_B])

    fig.update_layout(
        title="Trajectories in 3D Shared Subspace",
        scene=dict(xaxis_title="PC 1", yaxis_title="PC 2", zaxis_title="PC 3"),
        width=400,
        height=400,
    )
    fig.show()

In [87]:
def get_trajectory_data(data):

    stiched_session = []
    for k in data.keys():
        stiched_session.append(data[k])
    stiched_session = np.concatenate(stiched_session)

    pca = PCA(n_components=3)
    pca_session = pca.fit_transform(stiched_session)
    n_timepoints = 50
    cond_A_data = pca_session[:, 0:n_timepoints]
    cond_B_data = pca_session[:, n_timepoints:]

    plot_trajectories(cond_A_data, cond_B_data)

In [59]:
def rearrange_distance_matrix(matrix):

    final_structure = []
    n_sessions = len(matrix[0][0])
    n_eids = len(matrix)
    for c in range(2):
        session_layer = []
        for s in range(n_sessions):

            neurons_across_eids = [matrix[e][c][s] for e in range(n_eids)]
            all_neurons_combined = np.concatenate(neurons_across_eids)
            session_layer.append(all_neurons_combined)
        final_structure.append(session_layer)

    final_array = np.array(final_structure)
    return final_array

In [57]:
files = np.sort(glob("../data/generated/manifold/*.pkl"))
pseudo_files = np.sort(glob("../data/generated/manifold/pseudosession_metrics/*.pkl"))

In [ ]:
for idx in range(len(files)):
    fname = files[idx]
    fname_pseudo = pseudo_files[idx]
    with open(fname, "rb") as f:
        true_data = pkl.load(f)
    with open(fname_pseudo, "rb") as f:
        pseudo_data = pkl.load(f)
    total_distance_true, distance_matrix_true = process_eids(true_data)
    distance_matrix_true = np.asarray(distance_matrix_true).squeeze()
    distance_matrix_pseudo = np.asarray(pseudo_data["distance_matrix"])
    # totalds_pseudo = rearrange_distance_matrix(pseudo_data["total_distance_pseudosession"])
    totalds_true = np.asarray(total_distance_true)

    fig, ax = plt.subplots(ncols=2, figsize=(10, 4))
    # ax[0].plot(np.mean(distance_matrix_true, axis=0), "red")
    # ax[0].plot(np.median(np.mean(distance_matrix_pseudo, axis=(0)), axis=0), "black")

    sns.violinplot(totalds_true.squeeze(), ax=ax[1], palette=["green", "red"], alpha=0.75)
    sns.despine()

In [152]:
true_data["1a507308-c63a-4e02-8f32-3239a07dc578"].shape

(10, 100)

In [153]:
w.shape

(14, 2, 1)

In [158]:
from scipy.spatial.distance import cdist, cosine

In [ ]:
def analyze_stitched_manifold(pickle_path, n_timepoints=50, soft_norm_factor=5.0):
    """
    Loads session PSTHs, stitches them into a supersession, normalizes,
    and computes state-space geometry and trajectory metrics.
    """

    with open(pickle_path, "rb") as f:
        region_data = pkl.load(f)

    neuron_blocks = [psth_matrix for session_id, psth_matrix in region_data.items()]
    supersession = np.vstack(neuron_blocks)

    neuron_ranges = np.ptp(supersession, axis=1, keepdims=True)
    supersession_norm = supersession / (neuron_ranges + soft_norm_factor)

    cond_correct = supersession_norm[:, :n_timepoints]
    cond_incorrect = supersession_norm[:, n_timepoints:]
    centroid_correct = np.mean(cond_correct, axis=1)
    centroid_incorrect = np.mean(cond_incorrect, axis=1)
    real_similarity = 1 - cosine(centroid_correct, centroid_incorrect)

    len_correct = np.sum(np.linalg.norm(np.diff(cond_correct, axis=1), axis=0))
    len_incorrect = np.sum(np.linalg.norm(np.diff(cond_incorrect, axis=1), axis=0))

    path_length_difference = len_correct - len_incorrect

    return {
        "N_total_neurons": supersession.shape[0],
        "centroid_distance": real_similarity,
        "path_length_correct": len_correct,
        "path_length_incorrect": len_incorrect,
        "path_length_diff": path_length_difference,
    }

(306, 100)


In [157]:
results

{'N_total_neurons': 306,
 'cross_temporal_euclidean': array([[0.18603514, 0.18558202, 0.19052031, ..., 0.18243038, 0.18919858,
         0.18942273],
        [0.17755709, 0.18668346, 0.19485566, ..., 0.17023793, 0.18907283,
         0.16459702],
        [0.17658341, 0.18835415, 0.179256  , ..., 0.1936244 , 0.19156589,
         0.18974514],
        ...,
        [0.19819308, 0.21821806, 0.18633832, ..., 0.18076313, 0.18639629,
         0.15791047],
        [0.18708953, 0.1972928 , 0.1888957 , ..., 0.17304434, 0.17280368,
         0.15833304],
        [0.17529186, 0.19662083, 0.17951722, ..., 0.17375594, 0.1734495 ,
         0.16777431]], shape=(50, 50)),
 'cross_temporal_cosine': array([[0.74694982, 0.75361795, 0.72660725, ..., 0.72664734, 0.70098174,
         0.74503571],
        [0.77946387, 0.76047598, 0.7278622 , ..., 0.7775422 , 0.72149584,
         0.81537131],
        [0.7764413 , 0.75032655, 0.76319131, ..., 0.70074257, 0.7025619 ,
         0.74837773],
        ...,
        [0.718